In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Project/lhr.csv')

In [7]:
for c in df.columns:
    print(c)

datetime
tempmax
tempmin
temp
feelslikemax
feelslikemin
feelslike
dew
humidity
precip
precipprob
precipcover
snow
snowdepth
windgust
windspeed
winddir
sealevelpressure
cloudcover
visibility
solarradiation
solarenergy
uvindex
severerisk
moonphase
pm2.5
CO
NO2
O3
SO2
BCANGSTR
BCCMASS
BCEXTTAU
BCFLUXU
BCFLUXV
BCSCATAU
BCSMASS
DMSCMASS
DMSSMASS
DUANGSTR
DUCMASS
DUCMASS25
DUEXTT25
DUEXTTAU
DUFLUXU
DUFLUXV
DUSCAT25
DUSCATAU
DUSMASS
DUSMASS25
OCANGSTR
OCCMASS
OCEXTTAU
OCFLUXU
OCFLUXV
OCSCATAU
OCSMASS
SO2CMASS
SO2SMASS
SO4CMASS
SO4SMASS
SSANGSTR
SSCMASS
SSCMASS25
SSEXTT25
SSEXTTAU
SSFLUXU
SSFLUXV
SSSCAT25
SSSCATAU
SSSMASS
SSSMASS25
SUANGSTR
SUEXTTAU
SUFLUXU
SUFLUXV
SUSCATAU
TOTANGSTR
TOTEXTTAU
TOTSCATAU
Var_BCANGSTR
Var_BCCMASS
Var_BCEXTTAU
Var_BCFLUXU
Var_BCFLUXV
Var_BCSCATAU
Var_BCSMASS
Var_DMSCMASS
Var_DMSSMASS
Var_DUANGSTR
Var_DUCMASS
Var_DUCMASS25
Var_DUEXTT25
Var_DUEXTTAU
Var_DUFLUXU
Var_DUFLUXV
Var_DUSCAT25
Var_DUSCATAU
Var_DUSMASS
Var_DUSMASS25
Var_OCANGSTR
Var_OCCMASS
Var_OCEXTTAU
Var

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load all city datasets
cities = ['lhr', 'isb', 'khr', 'phr']
dfs = []
for city in cities:
    df_city = pd.read_csv(f"project/{city}.csv", parse_dates=['datetime'])
    df_city['City'] = city
    dfs.append(df_city)
    
master_df = pd.concat(dfs).sort_values('datetime').reset_index(drop=True)

# Feature Selection - Prioritize key variables
selected_features = [
    'datetime', 'City', 'temp', 'humidity', 'windspeed', 'winddir', 
    'precip', 'cloudcover', 'pm2.5', 'O3', 'DUEXTTAU', 'BCEXTTAU',
    'OCEXTTAU', 'SO4CMASS', 'TOTEXTTAU', 'sealevelpressure'
]

df = master_df[selected_features].copy()

# Create meteorological extremes
df['Heatwave'] = (df.groupby('City')['temp'].transform(lambda x: x > x.quantile(0.9))).astype(int)
df['Stagnation'] = (df.groupby('City')['windspeed'].transform(lambda x: x < x.quantile(0.1))).astype(int)
df['High_Dust'] = (df.groupby('City')['DUEXTTAU'].transform(lambda x: x > x.quantile(0.95))).astype(int)

# Create temporal features
df['Month'] = df['datetime'].dt.month
df['Year'] = df['datetime'].dt.year

# Handle missing data
# df.interpolate(method='time', inplace=True)
df.dropna(inplace=True)

In [4]:
# Define compound events
df['Heatwave_Stagnation'] = (df['Heatwave'] & df['Stagnation']).astype(int)
df['Dust_LowWind'] = (df['High_Dust'] & df['Stagnation']).astype(int)

# Detect events using dynamic thresholds
def detect_events(group):
    group['O3_Event'] = ((group['Heatwave_Stagnation'] == 1) & 
                         (group['O3'] > group['O3'].quantile(0.9)))
    group['PM_Event'] = ((group['Dust_LowWind'] == 1) & 
                        (group['pm2.5'] > group['pm2.5'].quantile(0.9)))
    return group

df = df.groupby('City').apply(detect_events)

C:\Users\tipua\AppData\Local\Temp\ipykernel_9584\1551094657.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('City').apply(detect_events)


In [9]:
# Define shock labels (sudden 60% increase from previous month)
def create_shock_label(group, pollutant, threshold):
    shock = (group[pollutant].pct_change() > 0.6) & (group[pollutant] > threshold)
    group[f'{pollutant}_Shock'] = shock.astype(int)
    return group
df = df.drop(columns=['City'])
df = df.groupby('City', group_keys=False).apply(create_shock_label, 'pm2.5', 50)
df = df.groupby('City', group_keys=False).apply(create_shock_label, 'O3', 60)


# Feature engineering for prediction
lags = [1, 2, 3]  # 1-3 month lags
for lag in lags:
    df[f'pm2.5_lag{lag}'] = df.groupby('City')['pm2.5'].shift(lag)
    df[f'O3_lag{lag}'] = df.groupby('City')['O3'].shift(lag)
    
df['Aerosol_Index'] = df['DUEXTTAU'] + df['BCEXTTAU'] + df['OCEXTTAU']
df['Stability_Index'] = df['sealevelpressure'] / (df['windspeed'] + 1e-5)

# Final cleaning
df.dropna(inplace=True)

In [10]:
df

datetime       temp   humidity  windspeed  \
City                                                                  
isb  31  2019-08-31 00:00:00+05:00  80.054839  75.732258   6.419355   
     32  2019-09-30 00:00:00+05:00  81.740000  67.780000   9.823333   
     38  2019-10-31 00:00:00+05:00  71.970968  60.819355   9.048387   
     42  2019-11-30 00:00:00+05:00  60.123333  71.110000   9.986667   
     47  2019-12-31 00:00:00+05:00  49.964516  71.177419   5.470968   
...                            ...        ...        ...        ...   
phr  270 2024-08-31 00:00:00+05:00  85.612903  74.351613  17.893548   
     272 2024-09-30 00:00:00+05:00  85.273333  63.140000  14.856667   
     278 2024-10-31 00:00:00+05:00  76.183871  60.332258  11.803226   
     280 2024-11-30 00:00:00+05:00  65.326667  60.783333   6.263333   
     287 2024-12-31 00:00:00+05:00  53.058065  44.174194   6.880645   

             winddir    precip  cloudcover       pm2.5        O3  DUEXTTAU  \
City                                                                         
isb  31   169.493548  1.339355   47.329032   29.424365  0.128850  0.202059   
     32   171.786667  0.528967   32.593333   41.776930  0.124633  0.234348   
     38   204.441935  0.608645   34.764516   49.471009  0.128880  0.126918   
     42   142.753333  0.972700   53.870000   56.058273  0.131495  0.072533   
     47   132.912903  0.179323   25.748387  105.741935  0.139177  0.026159   
...              ...       ...         ...         ...       ...       ...   
phr  270  131.803226  0.286871   64.009677   41.902020  0.129593  0.254706   
     272  119.650000  0.128833   31.433333   59.609212  0.127728  0.163430   
     278  167.277419  0.011290   23.503226   98.161998  0.124557  0.099503   
     280  155.980000  0.028767   33.460000  130.928030  0.128477  0.035566   
     287  219.883871  0.000000   33.570968  182.430654  0.135535  0.034267   

          ...  pm2.5_Shock  O3_Shock  pm2.5_lag1   O3_lag1  pm2.5_lag2  \
City      ...                                                            
isb  31   ...            0         0   33.956741  0.129441   31.511043   
     32   ...            0         0   29.424365  0.128850   33.956741   
     38   ...            0         0   41.776930  0.124633   29.424365   
     42   ...            0         0   49.471009  0.128880   41.776930   
     47   ...            1         0   56.058273  0.131495   49.471009   
...       ...          ...       ...         ...       ...         ...   
phr  270  ...            0         0   51.945384  0.132173   51.360127   
     272  ...            0         0   41.902020  0.129593   51.945384   
     278  ...            1         0   59.609212  0.127728   41.902020   
     280  ...            0         0   98.161998  0.124557   59.609212   
     287  ...            0         0  130.928030  0.128477   98.161998   

           O3_lag2  pm2.5_lag3   O3_lag3  Aerosol_Index  Stability_Index  
City                                                                      
isb  31   0.134441   26.580717  0.139556       0.274900       156.106289  
     32   0.129441   31.511043  0.134441       0.339586       102.447470  
     38   0.128850   33.956741  0.129441       0.212487       112.013067  
     42   0.124633   29.424365  0.128850       0.167367       101.831340  
     47   0.128880   41.776930  0.124633       0.094438       186.310980  
...            ...         ...       ...            ...              ...  
phr  270  0.139475   43.601587  0.139927       0.316968        56.087223  
     272  0.132173   51.360127  0.139475       0.233025        67.674848  
     278  0.129593   51.945384  0.132173       0.199068        85.738927  
     280  0.127728   41.902020  0.129593       0.148393       162.253599  
     287  0.124557   59.609212  0.127728       0.107273       148.031665  

[243 rows x 34 columns]